=============================================================================
MISSOURI VOTER RESOURCE ALLOCATION PROJECT
=============================================================================
Course: CAPS 5576 - Analytics Applications
Team: Group 1
Project: Prescriptive and Predictive Voter Resource Allocation
Focus: Missouri Presidential Elections (2016, 2020, 2024)
=============================================================================

COLLABORATION NOTE:
-------------------
This notebook was initially built in an individual environment (SNOWBEARAIR_DB).
Once it was validated, it was ported here to a shared team GitHub account
where all 4 team members can collaborate.

=============================================================================

FILES REQUIRED (upload as notebook assets before execution):
------------------------------------------------------------

ELECTION DATA (3 files):
- MO 2016 Election Results.csv
- MO 2020 Election Results.csv
- MO 2024 Election Results.csv

CENSUS DATA - 2016 (5 files):
- MO 2016 Census Income.csv
- MO 2016 Census Education.csv
- MO 2016 Census Race.csv
- MO 2016 Census Commute.csv
- MO 2016 Census Sex by Age.csv

CENSUS DATA - 2020 (5 files):
- MO 2020 Census Income.csv
- MO 2020 Census Education.csv
- MO 2020 Census Race.csv
- MO 2020 Census Commute.csv
- MO 2020 Census Sex by Age.csv

CENSUS DATA - 2024 (5 files):
- MO 2024 Census Income.csv
- MO 2024 Census Education.csv
- MO 2024 Census Race.csv
- MO 2024 Census Commute.csv
- MO 2024 Census Sex by Age.csv

POLLING LOCATION DATA (1 file):
- MO 2020 Polling Locations.csv

SHAPEFILE DATA (5 files):
- MO 2020 Precincts.shp
- MO 2020 Precincts.dbf
- MO 2020 Precincts.shx
- MO 2020 Precincts.prj
- MO 2020 Precincts.cpg

TOTAL: 24 files

=============================================================================

NOTEBOOK STRUCTURE:
-------------------
SECTION 1: Documentation & Setup

SECTION 2: Data Ingestion

SECTION 3: Data Cleaning - Election Data

SECTION 4: Data Cleaning - Census Data

SECTION 5: Data Cleaning - Polling & Shapefile

SECTION 6: Write Staging Tables to Snowflake

SECTION 7: SQL Aggregations & JOINs

SECTION 8: Data Quality Validation

SECTION 9: Exploratory Data Analysis

SECTION 10: Geospatial Analysis - inserted for David's geospatial idea

SECTION 11: Summary & Next Steps

=============================================================================

# Missouri Voter Resource Allocation Project

## Project Goal
Analyze historical presidential election results alongside demographic and geographic data to evaluate how polling resources could potentially be allocated more effectively across Missouri precincts.

## Analysis Focus
The analysis focuses on **three presidential election cycles**: 2016, 2020, and 2024.

Presidential election years were chosen because they produce the **highest voter turnout** and the **most consistent statewide participation**. Focusing on presidential election cycles provides a clearer signal for modeling voter demand and analyzing polling resource allocation across precincts.

## Data Architecture
The project integrates several categories of data:
- **Presidential election results** (precinct-level) - 2016, 2020, 2024
- **Census demographic data** (county-level) - ACS 5-Year Estimates aligned to each election
- **Polling location data** - Physical polling places by precinct
- **Precinct boundaries** (optional) - Geographic shapefiles for spatial analysis

## Programming Paradigms
- **Imperative (Python):** Data loading, cleaning, transformations
- **Declarative (SQL):** Joins, aggregations, analytical queries

## Coding Standards
- snake_case naming with meaningful variable names
- One output per cell
- Explanations in Markdown cells (not inline comments)
- Pandas method chaining
- List comprehensions over loops

# Data Sources

## Election Results Data

**Source:** OpenElections Project (GitHub)  
https://github.com/openelections/openelections-data-mo

**Files:**
- MO 2016 Election Results.csv (128,859 rows)
- MO 2020 Election Results.csv (132,123 rows)
- MO 2024 Election Results.csv (190,270 rows)

These datasets contain precinct-level vote totals for each candidate and office. Each row represents the vote total for one candidate within a specific precinct.

**Schema Note:** The 2020 and 2024 files include a `precinct_code` column not present in 2016. This column is dropped during preprocessing to maintain a consistent schema.

## Census Demographic Data

**Source:** U.S. Census Bureau – American Community Survey (ACS) 5-Year Estimates  
https://data.census.gov

Demographic data was aligned with each presidential election year:
- **2016 Election** → ACS 2012-2016
- **2020 Election** → ACS 2016-2020
- **2024 Election** → ACS 2020-2024

**Tables Used (5 per year = 15 files total):**
- B01001 – Sex by Age
- B02001 – Race
- B08301 – Commuting / Transportation to Work
- B15003 – Educational Attainment
- B19013 – Median Household Income

All ACS datasets are at the **county level** for Missouri.

## Polling Location Data

**Source:** MIT Election Data and Science Lab  
https://electionlab.mit.edu/data

**File:** MO 2020 Polling Locations.csv (14,354 rows)

Contains polling locations associated with Missouri precincts. Although the data represents the 2020 election cycle, polling locations generally change slowly, making the dataset suitable for analysis across nearby election years.

## Precinct Geographic Files

**Source:** U.S. Census TIGER/Line Shapefiles  

**Files:** MO 2020 Precincts (.shp, .dbf, .shx, .prj, .cpg)

These files represent Missouri Voting Tabulation District (precinct) boundaries. They may be used for:
- Mapping turnout geographically
- Spatial analysis of polling locations
- Calculating distances between voters and polling places

# Data Preprocessing Requirements

## Election Data Preprocessing

1. **Add election year column** - Enables combining datasets from different cycles
2. **Remove precinct_code column** - Not present in 2016, dropped from 2020/2024
3. **Normalize precinct names** - Uppercase, remove special characters, trim whitespace
4. **Normalize county names** - Uppercase and trim whitespace
5. **Filter to Presidential results only** - Focus analysis on highest-turnout races

## Census Data Preprocessing

1. **Remove header artifact row** - ACS exports include a metadata row where GEO_ID = "Geography"
2. **Remove blank export columns** - Drop any "Unnamed" columns from export artifacts
3. **Extract county name** - Parse from NAME field, removing " County, Missouri" suffix
4. **Add census year column** - Track which ACS vintage the data comes from
5. **Convert to numeric types** - Ensure proper data types for analysis

## Data Preprocessing

### Election Data Preprocessing
Before combining election datasets, several preprocessing steps are performed.

Add election year column:

```python
df2016["year"] = 2016
df2020["year"] = 2020
df2024["year"] = 2024
```

Remove precinct_code where present:

```python
df = df.drop(columns=["precinct_code"], errors="ignore")
```

Normalize precinct names:

```python
df["precinct_clean"] = (
    df["precinct"]
    .astype(str)
    .str.upper()
    .str.replace("#","")
    .str.replace("  "," ")
    .str.strip()
)
```

### Census Data Preprocessing
Census datasets exported from data.census.gov contain some artifacts that should be cleaned.

Remove header artifact row:

```python
df = df[df["GEO_ID"] != "Geography"]
```

Remove blank export columns:

```python
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
```

Extract county name:

```python
df["county_clean"] = (
    df["NAME"]
    .str.replace(", Missouri", "", regex=False)
    .str.replace(" County", "", regex=False)
    .str.upper()
)
```

**Important – St. Louis County vs. St. Louis City:**
Missouri has both "St. Louis County" (FIPS 29189) and "St. Louis city" (FIPS 29510), which is an independent city that is not part of any county. The cleaning process preserves this distinction by only removing " County" from county names, not " city". This results in:

- St. Louis County → "ST. LOUIS"
- St. Louis city → "ST. LOUIS CITY"

**Warning:** If both entities are cleaned to the same name, JOINs between election and census data will produce duplicate rows (a cartesian product), leading to inflated row counts in analytical tables.

Add census year column:

```python
df["census_year"] = 2016  # or 2020 or 2024
```

## Snowflake Implementation Notes

### Column Naming and Case Sensitivity

Snowflake handles column names differently depending on how tables are created:

**Staging Tables (created via Python/Snowpark):**
- Column names are lowercase (e.g., `year`, `county_clean`, `votes`)
- Must use double quotes in SQL to reference them: `"year"`, `"county_clean"`

**Analytical Tables (created via SQL):**
- Use uppercase aliases when creating: `SELECT "year" AS YEAR, "county_clean" AS COUNTY`
- This allows unquoted references in downstream SQL: `WHERE YEAR = 2020`

### Snowflake Table Structure

**Staging Tables (loaded via Python):**

STG_ELECTION_RESULTS – Combined presidential election results (all years)

STG_CENSUS_INCOME – Median household income by county

STG_CENSUS_EDUCATION – Educational attainment by county

STG_CENSUS_RACE – Race demographics by county

STG_CENSUS_COMMUTE – Commuting patterns by county

STG_CENSUS_SEX_AGE – Sex and age demographics by county

STG_POLLING_LOCATIONS – Polling place locations

**Analytical Tables (created via SQL):**

PRECINCT_TURNOUT – Aggregated votes by precinct and year

COUNTY_TURNOUT – Aggregated votes by county and year

COUNTY_TURNOUT_TREND – Pivoted view with all years side by side

COUNTY_CENSUS – Combined census demographics with all years

COUNTY_ANALYSIS – Joined turnout and census data for analysis

COUNTY_POLLING_SUMMARY – Polling locations aggregated by county

COUNTY_VIZ_EXPORT – Final export table for visualization tools

### Expected Row Counts

| Table | Expected Rows | Notes |
|-------|---------------|-------|
| STG_ELECTION_RESULTS | ~70,000 | Presidential votes only |
| STG_CENSUS_* | 345 each | 115 counties × 3 years |
| STG_POLLING_LOCATIONS | ~14,000 | 2020 polling places |
| PRECINCT_TURNOUT | ~10,000 | Precincts × 3 years |
| COUNTY_TURNOUT | 348 | 116 counties × 3 years |
| COUNTY_TURNOUT_TREND | 117 | One row per county |
| COUNTY_CENSUS | 345 | 115 counties × 3 years |
| COUNTY_ANALYSIS | 117 | One row per county |
| COUNTY_POLLING_SUMMARY | 116 | One row per county |

Note: Missouri has 114 counties plus the independent city of St. Louis, for a total of 115 county-level entities. Some election data may show 116 counties due to Kansas City reporting.

## Verified Results

Statewide presidential election totals match official Missouri results:

| Year | Total Votes | Republican % | Democrat % |
|------|-------------|--------------|------------|
| 2016 | 2,808,298 | 56.78% | 38.14% |
| 2020 | 2,963,270 | 57.17% | 41.03% |
| 2024 | 2,995,327 | 58.49% | 40.08% |


In [2]:
# Library Imports and Snowflake Connection (VSCode version)import pandas as pdimport numpy as npimport warningswarnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)# Connect to Snowflakeimport snowflake.connectorconn = snowflake.connector.connect(    account='ytdayoz-nc71712',    user='5576GROUP1',  # <-- replace with your username    password='YOUR_PASSWORD'  # <-- replace with your password,  # <-- replace with your password    database='VOTER_PROJECT_DB',    schema='ANALYTICS',    warehouse='COMPUTE_WH')# Create a cursor for running queriescursor = conn.cursor()print("=" * 70)print("CONNECTED TO SNOWFLAKE")print("=" * 70)print("Database:  VOTER_PROJECT_DB")print("Schema:    ANALYTICS")print("Warehouse: COMPUTE_WH")print("=" * 70)

---
# SECTION 2: Data Ingestion
---

Load all raw data files into pandas DataFrames. Files are loaded exactly as downloaded to preserve raw source data. All transformations occur programmatically to ensure reproducibility.

In [ ]:
# =============================================================================# SKIP THIS CELL IN VSCODE - Data Already Loaded in Snowflake# =============================================================================# The election data is already in Snowflake table: STG_ELECTION_RESULTS# Only run this cell if you need to reload from CSV files.# # To run this cell, you would need the CSV files in your local directory:# - MO 2016 Election Results.csv# - MO 2020 Election Results.csv  # - MO 2024 Election Results.csvSKIP_CSV_LOADING = True  # Set to False if you have the CSV files locallyif not SKIP_CSV_LOADING:    # Load Election Results (All Three Years)        raw_election_2016 = pd.read_csv(        'MO 2016 Election Results.csv',         keep_default_na=False,         na_values=['']    )        raw_election_2020 = pd.read_csv(        'MO 2020 Election Results.csv',         keep_default_na=False,         na_values=[''],         low_memory=False    )        raw_election_2024 = pd.read_csv(        'MO 2024 Election Results.csv',         keep_default_na=False,         na_values=['']    )        print("=" * 70)    print("ELECTION DATA LOADED")    print("=" * 70)    print(f"2016: {raw_election_2016.shape[0]:>10,} rows × {raw_election_2016.shape[1]} columns")    print(f"2020: {raw_election_2020.shape[0]:>10,} rows × {raw_election_2020.shape[1]} columns")    print(f"2024: {raw_election_2024.shape[0]:>10,} rows × {raw_election_2024.shape[1]} columns")    print("=" * 70)

In [ ]:
# =============================================================================# SKIP THIS CELL IN VSCODE - Data Already Loaded in Snowflake# =============================================================================# Census data is already in Snowflake tables: STG_CENSUS_*# Only run this cell if you need to reload from CSV files.SKIP_CSV_LOADING = True  # Set to False if you have the CSV files locallyif not SKIP_CSV_LOADING:    # Load Census Data - 2016 (ACS 2012-2016)        raw_census_2016_income = pd.read_csv('MO 2016 Census Income.csv', keep_default_na=False, na_values=[''])    raw_census_2016_education = pd.read_csv('MO 2016 Census Education.csv', keep_default_na=False, na_values=[''])    raw_census_2016_race = pd.read_csv('MO 2016 Census Race.csv', keep_default_na=False, na_values=[''])    raw_census_2016_commute = pd.read_csv('MO 2016 Census Commute.csv', keep_default_na=False, na_values=[''])    raw_census_2016_sex_age = pd.read_csv('MO 2016 Census Sex by Age.csv', keep_default_na=False, na_values=[''])        print("=" * 70)    print("CENSUS DATA LOADED - 2016 (ACS 2012-2016)")    print("=" * 70)    print(f"Income:      {raw_census_2016_income.shape[0]:>4} rows × {raw_census_2016_income.shape[1]:>3} columns")    print(f"Education:   {raw_census_2016_education.shape[0]:>4} rows × {raw_census_2016_education.shape[1]:>3} columns")    print(f"Race:        {raw_census_2016_race.shape[0]:>4} rows × {raw_census_2016_race.shape[1]:>3} columns")    print(f"Commute:     {raw_census_2016_commute.shape[0]:>4} rows × {raw_census_2016_commute.shape[1]:>3} columns")    print(f"Sex by Age:  {raw_census_2016_sex_age.shape[0]:>4} rows × {raw_census_2016_sex_age.shape[1]:>3} columns")    print("=" * 70)

In [ ]:
# =============================================================================# SKIP THIS CELL IN VSCODE - Data Already Loaded in Snowflake# =============================================================================# Census data is already in Snowflake tables: STG_CENSUS_*# Only run this cell if you need to reload from CSV files.SKIP_CSV_LOADING = True  # Set to False if you have the CSV files locallyif not SKIP_CSV_LOADING:    # Load Census Data - 2020 (ACS 2016-2020)        raw_census_2020_income = pd.read_csv('MO 2020 Census Income.csv', keep_default_na=False, na_values=[''])    raw_census_2020_education = pd.read_csv('MO 2020 Census Education.csv', keep_default_na=False, na_values=[''])    raw_census_2020_race = pd.read_csv('MO 2020 Census Race.csv', keep_default_na=False, na_values=[''])    raw_census_2020_commute = pd.read_csv('MO 2020 Census Commute.csv', keep_default_na=False, na_values=[''])    raw_census_2020_sex_age = pd.read_csv('MO 2020 Census Sex by Age.csv', keep_default_na=False, na_values=[''])        print("=" * 70)    print("CENSUS DATA LOADED - 2020 (ACS 2016-2020)")    print("=" * 70)    print(f"Income:      {raw_census_2020_income.shape[0]:>4} rows × {raw_census_2020_income.shape[1]:>3} columns")    print(f"Education:   {raw_census_2020_education.shape[0]:>4} rows × {raw_census_2020_education.shape[1]:>3} columns")    print(f"Race:        {raw_census_2020_race.shape[0]:>4} rows × {raw_census_2020_race.shape[1]:>3} columns")    print(f"Commute:     {raw_census_2020_commute.shape[0]:>4} rows × {raw_census_2020_commute.shape[1]:>3} columns")    print(f"Sex by Age:  {raw_census_2020_sex_age.shape[0]:>4} rows × {raw_census_2020_sex_age.shape[1]:>3} columns")    print("=" * 70)

In [ ]:
# =============================================================================# SKIP THIS CELL IN VSCODE - Data Already Loaded in Snowflake# =============================================================================# Census data is already in Snowflake tables: STG_CENSUS_*# Only run this cell if you need to reload from CSV files.SKIP_CSV_LOADING = True  # Set to False if you have the CSV files locallyif not SKIP_CSV_LOADING:    # Load Census Data - 2024 (ACS 2020-2024)        raw_census_2024_income = pd.read_csv('MO 2024 Census Income.csv', keep_default_na=False, na_values=[''])    raw_census_2024_education = pd.read_csv('MO 2024 Census Education.csv', keep_default_na=False, na_values=[''])    raw_census_2024_race = pd.read_csv('MO 2024 Census Race.csv', keep_default_na=False, na_values=[''])    raw_census_2024_commute = pd.read_csv('MO 2024 Census Commute.csv', keep_default_na=False, na_values=[''])    raw_census_2024_sex_age = pd.read_csv('MO 2024 Census Sex by Age.csv', keep_default_na=False, na_values=[''])        print("=" * 70)    print("CENSUS DATA LOADED - 2024 (ACS 2020-2024)")    print("=" * 70)    print(f"Income:      {raw_census_2024_income.shape[0]:>4} rows × {raw_census_2024_income.shape[1]:>3} columns")    print(f"Education:   {raw_census_2024_education.shape[0]:>4} rows × {raw_census_2024_education.shape[1]:>3} columns")    print(f"Race:        {raw_census_2024_race.shape[0]:>4} rows × {raw_census_2024_race.shape[1]:>3} columns")    print(f"Commute:     {raw_census_2024_commute.shape[0]:>4} rows × {raw_census_2024_commute.shape[1]:>3} columns")    print(f"Sex by Age:  {raw_census_2024_sex_age.shape[0]:>4} rows × {raw_census_2024_sex_age.shape[1]:>3} columns")    print("=" * 70)

In [ ]:
# Load Polling Locations Data

raw_polling_locations = pd.read_csv(
    'MO 2020 Polling Locations.csv', 
    keep_default_na=False, 
    na_values=['']
)

print("=" * 70)
print("POLLING LOCATIONS DATA LOADED")
print("=" * 70)
print(f"Rows:    {raw_polling_locations.shape[0]:,}")
print(f"Columns: {raw_polling_locations.shape[1]}")
print(f"Columns: {raw_polling_locations.columns.tolist()}")
print("=" * 70)

In [ ]:
# =============================================================================# GEOSPATIAL DATA - Load Missouri Precinct Shapefile# =============================================================================# This cell loads the precinct boundaries shapefile for geospatial analysis.# # REQUIRED FILES (must be in the same directory or provide full path):# - MO 2020 Precincts.shp# - MO 2020 Precincts.dbf# - MO 2020 Precincts.shx# - MO 2020 Precincts.prj# - MO 2020 Precincts.cpg# UPDATE THIS PATH to where your shapefile is located:SHAPEFILE_PATH = 'MO 2020 Precincts.shp'  # <-- Update this pathtry:    precincts_gdf = gpd.read_file(SHAPEFILE_PATH)    print("=" * 70)    print("SHAPEFILE LOADED SUCCESSFULLY")    print("=" * 70)    print(f"Total precincts: {len(precincts_gdf):,}")    print(f"CRS: {precincts_gdf.crs}")    print(f"Columns: {list(precincts_gdf.columns)}")    print("=" * 70)    precincts_gdf.head()except FileNotFoundError:    print("=" * 70)    print("SHAPEFILE NOT FOUND")    print("=" * 70)    print(f"Could not find: {SHAPEFILE_PATH}")    print("Please update SHAPEFILE_PATH to the correct location.")    print("=" * 70)    precincts_gdf = None

---
# SECTION 3: Data Cleaning - Election Data
---

Clean and standardize election results across all three years:
1. Add year column
2. Drop precinct_code (2020/2024 only)
3. Normalize precinct and county names
4. Filter to Presidential results only
5. Combine into single dataset

In [ ]:
# Define Precinct Name Normalization Function

def normalize_precinct_name(name):
    """
    Standardize precinct names for consistent joining across datasets.
    Missouri precinct names vary (e.g., 'Ward 1', 'WARD 1', 'Ward #1').
    """
    return (
        str(name)
        .upper()
        .replace('#', '')
        .replace('  ', ' ')
        .strip()
    )

print("normalize_precinct_name() function defined")

In [ ]:
# Clean and Filter Election Results (Presidential Only)

election_2016_clean = (
    raw_election_2016
    .query("office == 'President'")
    .assign(year=2016)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2020_clean = (
    raw_election_2020
    .query("office == 'President'")
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2020)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2024_clean = (
    raw_election_2024
    .loc[lambda x: x['office'] == 'President']
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2024)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

print("=" * 70)
print("PRESIDENTIAL ELECTION DATA CLEANED")
print("=" * 70)
print(f"2016 Presidential: {election_2016_clean.shape[0]:>8,} rows")
print(f"2020 Presidential: {election_2020_clean.shape[0]:>8,} rows")
print(f"2024 Presidential: {election_2024_clean.shape[0]:>8,} rows")
print("=" * 70)

In [ ]:
# Combine All Election Years into Single Dataset

all_elections_df = pd.concat(
    [election_2016_clean, election_2020_clean, election_2024_clean], 
    ignore_index=True
)

print("=" * 70)
print("COMBINED ELECTION DATASET")
print("=" * 70)
print(f"Total Rows:    {all_elections_df.shape[0]:,}")
print(f"Total Columns: {all_elections_df.shape[1]}")
print(f"Years:         {sorted(all_elections_df['year'].unique().tolist())}")
print(f"Counties:      {all_elections_df['county_clean'].nunique()}")
print(f"Parties:       {all_elections_df['party'].unique().tolist()}")
print("=" * 70)
all_elections_df.head(10)

---
# SECTION 4: Data Cleaning - Census Data
---

Clean census data for all three years. Each census year is processed identically:
1. Remove header artifact row (GEO_ID = "Geography")
2. Remove state-level totals
3. Extract county FIPS and clean county name
4. Add census year column
5. Rename ACS codes to readable names
6. Convert to numeric types
7. Calculate derived metrics (percentages)

In [ ]:
# Define Census Cleaning Functions

# NOTE: We do NOT remove " city" from county names to preserve the distinction
# between St. Louis County (FIPS 29189) and St. Louis city (FIPS 29510)

def clean_census_income(df, census_year):
    """Clean census income data for a given year."""
    return (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={'B19013_001E': 'median_household_income'})
        .assign(median_household_income=lambda x: pd.to_numeric(x['median_household_income'], errors='coerce'))
        [['census_year', 'county_fips', 'county_name', 'county_clean', 'median_household_income']]
        .reset_index(drop=True)
    )

def clean_census_education(df, census_year):
    """Clean census education data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B15003_001E': 'total_pop_25_plus',
            'B15003_017E': 'hs_diploma',
            'B15003_022E': 'bachelors_degree',
            'B15003_023E': 'masters_degree',
            'B15003_024E': 'professional_degree',
            'B15003_025E': 'doctorate_degree'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_pop_25_plus', 'hs_diploma', 'bachelors_degree', 'masters_degree', 'professional_degree', 'doctorate_degree']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_bachelors_plus'] = (
        (result['bachelors_degree'] + result['masters_degree'] + 
         result['professional_degree'] + result['doctorate_degree']) 
        / result['total_pop_25_plus'] * 100
    ).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_pop_25_plus', 'pct_bachelors_plus']]

def clean_census_race(df, census_year):
    """Clean census race data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B02001_001E': 'total_population',
            'B02001_002E': 'white_alone',
            'B02001_003E': 'black_alone'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_population', 'white_alone', 'black_alone']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_white'] = (result['white_alone'] / result['total_population'] * 100).round(2)
    result['pct_minority'] = ((result['total_population'] - result['white_alone']) / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'pct_white', 'pct_minority']]

def clean_census_commute(df, census_year):
    """Clean census commute data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B08301_001E': 'total_workers',
            'B08301_003E': 'drove_alone',
            'B08301_010E': 'public_transit',
            'B08301_019E': 'walked'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_workers', 'drove_alone', 'public_transit', 'walked']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_no_vehicle'] = ((result['public_transit'] + result['walked']) / result['total_workers'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_workers', 'pct_no_vehicle']]

def clean_census_sex_age(df, census_year):
    """Clean census sex by age data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .reset_index(drop=True)
    )
    
    for col in result.columns:
        if col.startswith('B01001'):
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['total_population'] = result['B01001_001E']
    
    male_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(7, 26)]
    female_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(31, 50)]
    
    male_18plus_cols = [c for c in male_18plus_cols if c in result.columns]
    female_18plus_cols = [c for c in female_18plus_cols if c in result.columns]
    
    result['voting_age_population'] = result[male_18plus_cols].sum(axis=1) + result[female_18plus_cols].sum(axis=1)
    result['pct_voting_age'] = (result['voting_age_population'] / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'voting_age_population', 'pct_voting_age']]

print("Census cleaning functions defined")

In [ ]:
# Clean Census Income Data (All Years)

census_income_2016 = clean_census_income(raw_census_2016_income, 2016)
census_income_2020 = clean_census_income(raw_census_2020_income, 2020)
census_income_2024 = clean_census_income(raw_census_2024_income, 2024)

all_census_income_df = pd.concat(
    [census_income_2016, census_income_2020, census_income_2024],
    ignore_index=True
)

print(f"Census Income cleaned: {all_census_income_df.shape[0]} rows ({all_census_income_df['census_year'].nunique()} years)")
all_census_income_df.head()

In [ ]:
# Clean Census Education Data (All Years)

census_education_2016 = clean_census_education(raw_census_2016_education, 2016)
census_education_2020 = clean_census_education(raw_census_2020_education, 2020)
census_education_2024 = clean_census_education(raw_census_2024_education, 2024)

all_census_education_df = pd.concat(
    [census_education_2016, census_education_2020, census_education_2024],
    ignore_index=True
)

print(f"Census Education cleaned: {all_census_education_df.shape[0]} rows ({all_census_education_df['census_year'].nunique()} years)")
all_census_education_df.head()

In [ ]:
# Clean Census Race Data (All Years)

census_race_2016 = clean_census_race(raw_census_2016_race, 2016)
census_race_2020 = clean_census_race(raw_census_2020_race, 2020)
census_race_2024 = clean_census_race(raw_census_2024_race, 2024)

all_census_race_df = pd.concat(
    [census_race_2016, census_race_2020, census_race_2024],
    ignore_index=True
)

print(f"Census Race cleaned: {all_census_race_df.shape[0]} rows ({all_census_race_df['census_year'].nunique()} years)")
all_census_race_df.head()

In [ ]:
# Clean Census Commute Data (All Years)

census_commute_2016 = clean_census_commute(raw_census_2016_commute, 2016)
census_commute_2020 = clean_census_commute(raw_census_2020_commute, 2020)
census_commute_2024 = clean_census_commute(raw_census_2024_commute, 2024)

all_census_commute_df = pd.concat(
    [census_commute_2016, census_commute_2020, census_commute_2024],
    ignore_index=True
)

print(f"Census Commute cleaned: {all_census_commute_df.shape[0]} rows ({all_census_commute_df['census_year'].nunique()} years)")
all_census_commute_df.head()

In [ ]:
# Clean Census Sex by Age Data (All Years)

census_sex_age_2016 = clean_census_sex_age(raw_census_2016_sex_age, 2016)
census_sex_age_2020 = clean_census_sex_age(raw_census_2020_sex_age, 2020)
census_sex_age_2024 = clean_census_sex_age(raw_census_2024_sex_age, 2024)

all_census_sex_age_df = pd.concat(
    [census_sex_age_2016, census_sex_age_2020, census_sex_age_2024],
    ignore_index=True
)

print(f"Census Sex/Age cleaned: {all_census_sex_age_df.shape[0]} rows ({all_census_sex_age_df['census_year'].nunique()} years)")
all_census_sex_age_df.head()

---
# SECTION 5: Data Cleaning - Polling Locations & Shapefile
---

Clean polling locations and prepare shapefile for potential geospatial analysis.

In [ ]:
# Clean Polling Locations Data

polling_locations_df = (
    raw_polling_locations
    .copy()
    .assign(county_clean=lambda x: x['county_name'].str.strip().str.upper())
    .assign(precinct_clean=lambda x: x['precinct_name'].apply(normalize_precinct_name))
    .assign(polling_address=lambda x: x['address'].str.strip())
)

print("=" * 70)
print("POLLING LOCATIONS CLEANED")
print("=" * 70)
print(f"Total Rows:           {polling_locations_df.shape[0]:,}")
print(f"Unique Counties:      {polling_locations_df['county_clean'].nunique()}")
print(f"Unique Precincts:     {polling_locations_df['precinct_clean'].nunique()}")
print(f"Unique Polling Places: {polling_locations_df['polling_place_id'].nunique()}")
print("=" * 70)
polling_locations_df.head()

In [ ]:
# Prepare Shapefile Data
# David, this could will not work until we get geopandas imported above
#   So, I've commented it out

# precincts_df = (
#     raw_precincts_gdf
#     .drop(columns=['geometry'])
#     .assign(county_fips=lambda x: '29' + x['COUNTYFP20'])
#     .assign(precinct_name=lambda x: x['NAME20'])
#     .assign(precinct_clean=lambda x: x['NAME20'].apply(normalize_precinct_name))
#     .assign(land_area_sqm=lambda x: x['ALAND20'])
#     .assign(water_area_sqm=lambda x: x['AWATER20'])
#     .assign(centroid_lat=lambda x: pd.to_numeric(x['INTPTLAT20'], errors='coerce'))
#     .assign(centroid_lon=lambda x: pd.to_numeric(x['INTPTLON20'], errors='coerce'))
#     [['county_fips', 'COUNTYFP20', 'precinct_name', 'precinct_clean', 
#       'GEOID20', 'land_area_sqm', 'water_area_sqm', 'centroid_lat', 'centroid_lon']]
#     .rename(columns={'COUNTYFP20': 'county_fips_3', 'GEOID20': 'geoid'})
# )

# print("=" * 70)
# print("SHAPEFILE DATA PREPARED (Tabular - No Geometry)")
# print("=" * 70)
# print(f"Precincts: {precincts_df.shape[0]:,}")
# print(f"Columns:   {precincts_df.columns.tolist()}")
# print("=" * 70)
# precincts_df.head()

---
# SECTION 6: Write Staging Tables to Snowflake
---

Write all cleaned DataFrames to Snowflake staging tables. These tables will be used for SQL-based aggregations and joins.

In [ ]:
# Write All Staging Tables to Snowflake

staging_tables = {
    'STG_ELECTION_RESULTS': all_elections_df,
    'STG_CENSUS_INCOME': all_census_income_df,
    'STG_CENSUS_EDUCATION': all_census_education_df,
    'STG_CENSUS_RACE': all_census_race_df,
    'STG_CENSUS_COMMUTE': all_census_commute_df,
    'STG_CENSUS_SEX_AGE': all_census_sex_age_df,
    'STG_POLLING_LOCATIONS': polling_locations_df
}

print("=" * 70)
print("WRITING STAGING TABLES TO SNOWFLAKE")
print("=" * 70)

for table_name, df in staging_tables.items():
    session.write_pandas(df, table_name, auto_create_table=True, overwrite=True)
    print(f"{table_name}: {len(df):,} rows")

print("=" * 70)
print("All staging tables written successfully!")
print("=" * 70)

## Snowflake Column Naming Convention

**Staging tables** (created via Python) have **lowercase** column names:
- Reference with double quotes in SQL: `"year"`, `"county_clean"`

**Analytical tables** (created via SQL) use **UPPERCASE** aliases:
- Created with: `SELECT "year" AS YEAR`
- Reference without quotes: `WHERE YEAR = 2020`

This convention simplifies downstream SQL queries.

---
# SECTION 7: SQL Aggregations & JOINs (Declarative)
---

Create analytical tables using SQL:
- Aggregate precinct-level votes to county level
- Pivot turnout by year
- Join election results with census demographics

In [ ]:
# Execute SQL: Create/Update Tablesql = """CREATE OR REPLACE TABLE PRECINCT_TURNOUT ASSELECT     "year" AS YEAR,    "county_clean" AS COUNTY,    "precinct_clean" AS PRECINCT,    SUM("votes") AS TOTAL_VOTES,    SUM(CASE WHEN "party" = 'REP' THEN "votes" ELSE 0 END) AS REPUBLICAN_VOTES,    SUM(CASE WHEN "party" = 'DEM' THEN "votes" ELSE 0 END) AS DEMOCRAT_VOTES,    SUM(CASE WHEN "party" NOT IN ('REP', 'DEM') OR "party" IS NULL THEN "votes" ELSE 0 END) AS OTHER_VOTES,    ROUND(SUM(CASE WHEN "party" = 'REP' THEN "votes" ELSE 0 END) / NULLIF(SUM("votes"), 0) * 100, 2) AS REPUBLICAN_PCT,    ROUND(SUM(CASE WHEN "party" = 'DEM' THEN "votes" ELSE 0 END) / NULLIF(SUM("votes"), 0) * 100, 2) AS DEMOCRAT_PCTFROM STG_ELECTION_RESULTSGROUP BY "year", "county_clean", "precinct_clean"ORDER BY "year", "county_clean", "precinct_clean";"""cursor.execute(sql)print("Table created/updated successfully")

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Aggregate to county level for each yearCREATE OR REPLACE TABLE COUNTY_TURNOUT ASSELECT     YEAR,    COUNTY,    COUNT(DISTINCT PRECINCT) AS PRECINCT_COUNT,    SUM(TOTAL_VOTES) AS TOTAL_VOTES,    SUM(REPUBLICAN_VOTES) AS REPUBLICAN_VOTES,    SUM(DEMOCRAT_VOTES) AS DEMOCRAT_VOTES,    SUM(OTHER_VOTES) AS OTHER_VOTES,    ROUND(SUM(REPUBLICAN_VOTES) / NULLIF(SUM(TOTAL_VOTES), 0) * 100, 2) AS REPUBLICAN_PCT,    ROUND(SUM(DEMOCRAT_VOTES) / NULLIF(SUM(TOTAL_VOTES), 0) * 100, 2) AS DEMOCRAT_PCTFROM PRECINCT_TURNOUTGROUP BY YEAR, COUNTYORDER BY YEAR, COUNTY;"""cursor.execute(sql)print("Table created/updated successfully")

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Pivot to show all years side by side for trend analysisCREATE OR REPLACE TABLE COUNTY_TURNOUT_TREND ASSELECT     COUNTY,    MAX(CASE WHEN YEAR = 2016 THEN PRECINCT_COUNT END) AS PRECINCTS_2016,    MAX(CASE WHEN YEAR = 2020 THEN PRECINCT_COUNT END) AS PRECINCTS_2020,    MAX(CASE WHEN YEAR = 2024 THEN PRECINCT_COUNT END) AS PRECINCTS_2024,    MAX(CASE WHEN YEAR = 2016 THEN TOTAL_VOTES END) AS VOTES_2016,    MAX(CASE WHEN YEAR = 2020 THEN TOTAL_VOTES END) AS VOTES_2020,    MAX(CASE WHEN YEAR = 2024 THEN TOTAL_VOTES END) AS VOTES_2024,    MAX(CASE WHEN YEAR = 2016 THEN REPUBLICAN_PCT END) AS REP_PCT_2016,    MAX(CASE WHEN YEAR = 2020 THEN REPUBLICAN_PCT END) AS REP_PCT_2020,    MAX(CASE WHEN YEAR = 2024 THEN REPUBLICAN_PCT END) AS REP_PCT_2024,    MAX(CASE WHEN YEAR = 2016 THEN DEMOCRAT_PCT END) AS DEM_PCT_2016,    MAX(CASE WHEN YEAR = 2020 THEN DEMOCRAT_PCT END) AS DEM_PCT_2020,    MAX(CASE WHEN YEAR = 2024 THEN DEMOCRAT_PCT END) AS DEM_PCT_2024FROM COUNTY_TURNOUTGROUP BY COUNTYORDER BY COUNTY;"""cursor.execute(sql)print("Table created/updated successfully")

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Combine all census tables and pivot to get one row per county with all yearsCREATE OR REPLACE TABLE COUNTY_CENSUS ASSELECT     i."county_clean" AS COUNTY,    i."county_fips" AS COUNTY_FIPS,    i."county_name" AS COUNTY_NAME,    i."census_year" AS CENSUS_YEAR,    i."median_household_income" AS MEDIAN_HOUSEHOLD_INCOME,    e."pct_bachelors_plus" AS PCT_BACHELORS_PLUS,    r."total_population" AS TOTAL_POPULATION,    r."pct_minority" AS PCT_MINORITY,    c."pct_no_vehicle" AS PCT_NO_VEHICLE,    s."voting_age_population" AS VOTING_AGE_POPULATION,    s."pct_voting_age" AS PCT_VOTING_AGEFROM STG_CENSUS_INCOME iLEFT JOIN STG_CENSUS_EDUCATION e     ON i."county_clean" = e."county_clean" AND i."census_year" = e."census_year"LEFT JOIN STG_CENSUS_RACE r     ON i."county_clean" = r."county_clean" AND i."census_year" = r."census_year"LEFT JOIN STG_CENSUS_COMMUTE c     ON i."county_clean" = c."county_clean" AND i."census_year" = c."census_year"LEFT JOIN STG_CENSUS_SEX_AGE s     ON i."county_clean" = s."county_clean" AND i."census_year" = s."census_year"ORDER BY i."county_clean", i."census_year";"""cursor.execute(sql)print("Table created/updated successfully")

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Join turnout trends with census data for comprehensive analysis tableCREATE OR REPLACE TABLE COUNTY_ANALYSIS ASSELECT     t.COUNTY,        -- Turnout by year    t.VOTES_2016,    t.VOTES_2020,    t.VOTES_2024,    t.REP_PCT_2016,    t.REP_PCT_2020,    t.REP_PCT_2024,    t.DEM_PCT_2016,    t.DEM_PCT_2020,    t.DEM_PCT_2024,        -- 2016 Census    c16.TOTAL_POPULATION AS POP_2016,    c16.VOTING_AGE_POPULATION AS VAP_2016,    c16.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2016,    c16.PCT_BACHELORS_PLUS AS EDU_2016,    c16.PCT_MINORITY AS MINORITY_2016,        -- 2020 Census    c20.TOTAL_POPULATION AS POP_2020,    c20.VOTING_AGE_POPULATION AS VAP_2020,    c20.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2020,    c20.PCT_BACHELORS_PLUS AS EDU_2020,    c20.PCT_MINORITY AS MINORITY_2020,    c20.PCT_NO_VEHICLE AS NO_VEHICLE_2020,        -- 2024 Census    c24.TOTAL_POPULATION AS POP_2024,    c24.VOTING_AGE_POPULATION AS VAP_2024,    c24.MEDIAN_HOUSEHOLD_INCOME AS INCOME_2024,    c24.PCT_BACHELORS_PLUS AS EDU_2024,    c24.PCT_MINORITY AS MINORITY_2024,        -- Calculated turnout rates    ROUND(t.VOTES_2016 / NULLIF(c16.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2016,    ROUND(t.VOTES_2020 / NULLIF(c20.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2020,    ROUND(t.VOTES_2024 / NULLIF(c24.VOTING_AGE_POPULATION, 0) * 100, 2) AS TURNOUT_PCT_2024FROM COUNTY_TURNOUT_TREND tLEFT JOIN COUNTY_CENSUS c16 ON t.COUNTY = c16.COUNTY AND c16.CENSUS_YEAR = 2016LEFT JOIN COUNTY_CENSUS c20 ON t.COUNTY = c20.COUNTY AND c20.CENSUS_YEAR = 2020LEFT JOIN COUNTY_CENSUS c24 ON t.COUNTY = c24.COUNTY AND c24.CENSUS_YEAR = 2024ORDER BY t.COUNTY;"""cursor.execute(sql)print("Table created/updated successfully")

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Aggregate polling locations by countyCREATE OR REPLACE TABLE COUNTY_POLLING_SUMMARY ASSELECT     "county_clean" AS COUNTY,    COUNT(DISTINCT "polling_place_id") AS UNIQUE_POLLING_PLACES,    COUNT(DISTINCT "precinct_clean") AS UNIQUE_PRECINCTS,    COUNT(*) AS TOTAL_RECORDS,    ROUND(COUNT(DISTINCT "precinct_clean") / NULLIF(COUNT(DISTINCT "polling_place_id"), 0), 2) AS PRECINCTS_PER_POLLING_PLACEFROM STG_POLLING_LOCATIONSGROUP BY "county_clean"ORDER BY PRECINCTS_PER_POLLING_PLACE DESC;"""cursor.execute(sql)print("Table created/updated successfully")

---
# SECTION 8: Data Quality Validation
---

Verify data integrity before proceeding with EDA:
- Row counts for all tables
- County matching validation
- Missing value checks

### Expected Row Counts

| Table | Expected Rows |
|-------|---------------|
| COUNTY_ANALYSIS | 117 (one per county) |
| COUNTY_TURNOUT | 348 (116 counties × 3 years) |
| COUNTY_CENSUS | 345 (115 counties × 3 years) |

In [ ]:
# Data Quality Summary - Table Row Counts

print("=" * 70)
print("DATA QUALITY SUMMARY - TABLE ROW COUNTS")
print("=" * 70)

staging_tables = [
    'STG_ELECTION_RESULTS', 'STG_CENSUS_INCOME', 'STG_CENSUS_EDUCATION',
    'STG_CENSUS_RACE', 'STG_CENSUS_COMMUTE', 'STG_CENSUS_SEX_AGE',
    'STG_POLLING_LOCATIONS'
]

analytical_tables = [
    'PRECINCT_TURNOUT', 'COUNTY_TURNOUT', 'COUNTY_TURNOUT_TREND',
    'COUNTY_CENSUS', 'COUNTY_ANALYSIS', 'COUNTY_POLLING_SUMMARY'
]

print("\nSTAGING TABLES:")
for table in staging_tables:
    count = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"   {table}: {count:,} rows")

print("\nANALYTICAL TABLES:")
for table in analytical_tables:
    count = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"   {table}: {count:,} rows")

print("=" * 70)

In [ ]:
# Execute SQL Querysql = """-- Check for county mismatches between election and census dataSELECT     'Election counties not in Census' AS validation_check,    COUNT(DISTINCT t.county) AS countFROM COUNTY_TURNOUT tLEFT JOIN COUNTY_CENSUS c ON t.county = c.countyWHERE c.county IS NULLUNION ALLSELECT     'Census counties not in Election' AS validation_check,    COUNT(DISTINCT c.county) AS countFROM COUNTY_CENSUS cLEFT JOIN COUNTY_TURNOUT t ON c.county = t.countyWHERE t.county IS NULL;"""df = pd.read_sql(sql, conn)df

---
# SECTION 9: Exploratory Data Analysis (EDA)
---

Analyze patterns in:
- Statewide turnout trends
- County-level turnout variations
- Demographic correlations
- Polling resource distribution

In [ ]:
# Execute SQL Querysql = """-- Statewide Turnout Summary by YearSELECT     YEAR,    COUNT(DISTINCT COUNTY) AS COUNTIES,    COUNT(DISTINCT PRECINCT) AS PRECINCTS,    SUM(TOTAL_VOTES) AS TOTAL_VOTES,    ROUND(SUM(REPUBLICAN_VOTES) / SUM(TOTAL_VOTES) * 100, 2) AS STATEWIDE_REP_PCT,    ROUND(SUM(DEMOCRAT_VOTES) / SUM(TOTAL_VOTES) * 100, 2) AS STATEWIDE_DEM_PCTFROM PRECINCT_TURNOUTGROUP BY YEARORDER BY YEAR;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- County Turnout Descriptive Statistics (2020)SELECT     '2020 County Stats' AS METRIC,    COUNT(*) AS N_COUNTIES,    ROUND(AVG(TOTAL_VOTES), 0) AS MEAN_VOTES,    ROUND(MEDIAN(TOTAL_VOTES), 0) AS MEDIAN_VOTES,    MIN(TOTAL_VOTES) AS MIN_VOTES,    MAX(TOTAL_VOTES) AS MAX_VOTES,    ROUND(STDDEV(TOTAL_VOTES), 0) AS STD_VOTESFROM COUNTY_TURNOUTWHERE YEAR = 2020;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Top 10 Counties by Total Votes (2020)SELECT     COUNTY,    TOTAL_VOTES,    REPUBLICAN_PCT,    DEMOCRAT_PCT,    PRECINCT_COUNTFROM COUNTY_TURNOUTWHERE YEAR = 2020ORDER BY TOTAL_VOTES DESCLIMIT 10;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Top 10 Counties by Turnout Rate (2020)SELECT     COUNTY,    TURNOUT_PCT_2020,    VOTES_2020,    VAP_2020 AS VOTING_AGE_POP,    POP_2020 AS TOTAL_POPFROM COUNTY_ANALYSISWHERE TURNOUT_PCT_2020 IS NOT NULLORDER BY TURNOUT_PCT_2020 DESCLIMIT 10;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Bottom 10 Counties by Turnout Rate (2020)SELECT     COUNTY,    TURNOUT_PCT_2020,    VOTES_2020,    VAP_2020 AS VOTING_AGE_POP,    POP_2020 AS TOTAL_POPFROM COUNTY_ANALYSISWHERE TURNOUT_PCT_2020 IS NOT NULLORDER BY TURNOUT_PCT_2020 ASCLIMIT 10;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Turnout Change 2016 to 2024SELECT     COUNTY,    VOTES_2016,    VOTES_2020,    VOTES_2024,    VOTES_2024 - VOTES_2016 AS VOTE_CHANGE,    ROUND((VOTES_2024 - VOTES_2016) / NULLIF(VOTES_2016, 0) * 100, 2) AS PCT_CHANGEFROM COUNTY_TURNOUT_TRENDWHERE VOTES_2016 IS NOT NULL AND VOTES_2024 IS NOT NULLORDER BY PCT_CHANGE DESCLIMIT 10;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Polling Resource Strain AnalysisSELECT     p.COUNTY,    p.UNIQUE_POLLING_PLACES,    p.UNIQUE_PRECINCTS,    p.PRECINCTS_PER_POLLING_PLACE,    a.POP_2020,    a.VOTES_2020,    a.TURNOUT_PCT_2020FROM COUNTY_POLLING_SUMMARY pLEFT JOIN COUNTY_ANALYSIS a ON p.COUNTY = a.COUNTYWHERE p.PRECINCTS_PER_POLLING_PLACE > 1ORDER BY p.PRECINCTS_PER_POLLING_PLACE DESCLIMIT 15;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """-- Demographics vs Turnout SnapshotSELECT     COUNTY,    POP_2020,    INCOME_2020,    EDU_2020 AS PCT_BACHELORS,    MINORITY_2020 AS PCT_MINORITY,    NO_VEHICLE_2020 AS PCT_NO_VEHICLE,    TURNOUT_PCT_2020,    REP_PCT_2020,    DEM_PCT_2020FROM COUNTY_ANALYSISWHERE TURNOUT_PCT_2020 IS NOT NULLORDER BY TURNOUT_PCT_2020 DESCLIMIT 20;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Correlation Analysis - Demographics vs Turnoutcounty_analysis_df = pd.read_sql('SELECT * FROM COUNTY_ANALYSIS', conn)print("=" * 70)print("CORRELATION MATRIX - DEMOGRAPHICS VS TURNOUT (2020)")print("=" * 70)correlation_cols = [    'INCOME_2020', 'EDU_2020', 'MINORITY_2020', 'NO_VEHICLE_2020',    'TURNOUT_PCT_2020', 'REP_PCT_2020', 'DEM_PCT_2020']valid_cols = [c for c in correlation_cols if c in county_analysis_df.columns]correlation_matrix = county_analysis_df[valid_cols].corr().round(3)print(correlation_matrix)print("=" * 70)

In [ ]:
# Key Findings Summary

print("=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print("""
1. STATEWIDE TRENDS:
   - Compare total turnout across 2016, 2020, 2024
   - Analyze party vote share changes over time
   
2. TURNOUT PATTERNS:
   - Identify high and low turnout counties
   - Examine turnout rate vs raw vote counts
   - Track changes in turnout over time

3. DEMOGRAPHIC CORRELATIONS:
   - Income vs turnout relationship
   - Education level impact on participation
   - Minority population voting patterns
   - Transportation access and voting

4. POLLING RESOURCES:
   - Counties with high precincts-per-polling-place
   - Cross-reference with population and turnout
   - Identify potential under-resourced areas

5. NEXT STEPS:
   - Predictive modeling for voter demand
   - Prescriptive recommendations for resource allocation
   - Geospatial visualization (David's section)
   - Final presentation preparation
""")
print("=" * 70)

---
# SECTION 10: Geospatial Analysis
---

**Owner: David**

This section is reserved for geospatial analysis using the precinct boundary shapefiles. Potential analyses include:
- Mapping voter turnout by precinct
- Visualizing polling location distribution
- Calculating distances between population centers and polling places
- Identifying geographic clusters of under-resourced areas

In [ ]:
# Geospatial Analysis - David (STUB)

# =============================================================================
# GEOSPATIAL ANALYSIS - DAVID
# =============================================================================
#
# This cell is a starting point for geospatial analysis.
# The shapefile data has been loaded and the tabular version is in STG_PRECINCTS.
#
# AVAILABLE DATA:
# ---------------
# - raw_precincts_gdf: GeoDataFrame with full geometry (loaded in Cell 12)
# - STG_PRECINCTS: Tabular precinct data with centroids (lat/lon)
#
# POTENTIAL ANALYSES:
# -------------------
# 1. Choropleth map of turnout by precinct
# 2. Polling location accessibility analysis
# 3. Distance calculations (voters to nearest polling place)
# 4. Geographic clustering of low-turnout areas
#
# SUGGESTED LIBRARIES:
# --------------------
# - geopandas (already imported)
# - folium (for interactive maps)
# - shapely (for geometry operations)
#
# =============================================================================

# David - add your geospatial analysis code below:

# print("=" * 70)
# print("GEOSPATIAL ANALYSIS - DAVID")
# print("=" * 70)
# print("Shapefile loaded: MO 2020 Precincts")
# print(f"Total precincts: {raw_precincts_gdf.shape[0]:,}")
# print(f"CRS: {raw_precincts_gdf.crs}")
# print(f"Geometry types: {raw_precincts_gdf.geom_type.unique().tolist()}")
print("=" * 70)
# 
# Example: View first few precincts
# raw_precincts_gdf[['NAME20', 'COUNTYFP20', 'ALAND20', 'INTPTLAT20', 'INTPTLON20']].head()

In [ ]:
# Execute SQL: Create/Update Tablesql = """-- Create Visualization Export TableCREATE OR REPLACE TABLE COUNTY_VIZ_EXPORT ASSELECT     a.COUNTY,    a.POP_2016, a.POP_2020, a.POP_2024,    a.VAP_2016, a.VAP_2020, a.VAP_2024,    a.VOTES_2016, a.VOTES_2020, a.VOTES_2024,    a.TURNOUT_PCT_2016, a.TURNOUT_PCT_2020, a.TURNOUT_PCT_2024,    a.REP_PCT_2016, a.REP_PCT_2020, a.REP_PCT_2024,    a.DEM_PCT_2016, a.DEM_PCT_2020, a.DEM_PCT_2024,    a.INCOME_2016, a.INCOME_2020, a.INCOME_2024,    a.EDU_2016, a.EDU_2020, a.EDU_2024,    a.MINORITY_2016, a.MINORITY_2020, a.MINORITY_2024,    a.NO_VEHICLE_2020,    p.UNIQUE_POLLING_PLACES,    p.UNIQUE_PRECINCTS,    p.PRECINCTS_PER_POLLING_PLACEFROM COUNTY_ANALYSIS aLEFT JOIN COUNTY_POLLING_SUMMARY p ON a.COUNTY = p.COUNTYORDER BY a.COUNTY;"""cursor.execute(sql)print("Table created/updated successfully")

---
# SECTION 11: Summary & Next Steps
---

## Data Pipeline Complete

### Staging Tables Created:
- `STG_ELECTION_RESULTS` - Combined presidential results (2016, 2020, 2024)
- `STG_CENSUS_INCOME` - Median household income (3 years)
- `STG_CENSUS_EDUCATION` - Educational attainment (3 years)
- `STG_CENSUS_RACE` - Race demographics (3 years)
- `STG_CENSUS_COMMUTE` - Transportation/commute patterns (3 years)
- `STG_CENSUS_SEX_AGE` - Age/sex distribution with VAP (3 years)
- `STG_POLLING_LOCATIONS` - Polling place locations (2020)
- `STG_PRECINCTS` - Precinct boundary data (2020)

### Analytical Tables Created:
- `PRECINCT_TURNOUT` - Precinct-level turnout by year
- `COUNTY_TURNOUT` - County-level turnout by year
- `COUNTY_TURNOUT_TREND` - Turnout pivoted across years
- `COUNTY_CENSUS` - Census demographics by year
- `COUNTY_ANALYSIS` - Master analysis table with all metrics
- `COUNTY_POLLING_SUMMARY` - Polling resource distribution
- `COUNTY_VIZ_EXPORT` - Export-ready for visualization tools

## Next Steps for Team:
1. **Predictive Modeling** - Build models to predict voter demand
2. **Prescriptive Analytics** - Recommend polling resource allocation
3. **Geospatial Analysis** - David's precinct mapping work
4. **Visualization** - Tableau/Flourish dashboards
5. **Final Presentation** - Synthesize findings

In [ ]:
# Final Status

print("=" * 70)
print("NOTEBOOK EXECUTION COMPLETE")
print("=" * 70)
print("""
DATA LOADED:
  • 3 Election years (2016, 2020, 2024) - Presidential results only
  • 15 Census files (5 tables × 3 years)
  • 1 Polling locations file (2020)
  • 1 Shapefile set (precinct boundaries)

TABLES CREATED: 14 total (8 staging + 6 analytical)

READY FOR:
  • Predictive modeling
  • Prescriptive analytics
  • Geospatial analysis (David)
  • Visualization exports
  
COLLABORATION NOTE:
  This notebook will be ported to a shared team GitHub account
  for full team collaboration.
""")
print("=" * 70)

---
# SECTION XX: Various Analyses
---

## Data Analyses (for temporary awareness in building our deliverables)



In [ ]:
# Execute SQL Querysql = """SELECT     a.COUNTY,    a.VOTES_2020,    p.UNIQUE_POLLING_PLACES,    ROUND(a.VOTES_2020 / NULLIF(p.UNIQUE_POLLING_PLACES, 0), 0) AS VOTERS_PER_POLLING_PLACE,    p.PRECINCTS_PER_POLLING_PLACE,    a.TURNOUT_PCT_2020FROM COUNTY_ANALYSIS aLEFT JOIN COUNTY_POLLING_SUMMARY p ON a.COUNTY = p.COUNTYWHERE a.VOTES_2020 IS NOT NULLORDER BY VOTERS_PER_POLLING_PLACE DESCLIMIT 15;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT     COUNTY,    INCOME_2020,    TURNOUT_PCT_2020,    NO_VEHICLE_2020FROM COUNTY_ANALYSISWHERE TURNOUT_PCT_2020 IS NOT NULLORDER BY INCOME_2020 DESC;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT     ROUND(AVG(TURNOUT_PCT_2020), 2) AS AVG_TURNOUT,    ROUND(MEDIAN(TURNOUT_PCT_2020), 2) AS MEDIAN_TURNOUT,    MIN(TURNOUT_PCT_2020) AS MIN_TURNOUT,    MAX(TURNOUT_PCT_2020) AS MAX_TURNOUTFROM COUNTY_ANALYSISWHERE TURNOUT_PCT_2020 IS NOT NULL;"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT DISTINCT "county_clean" FROM STG_POLLING_LOCATIONS WHERE "county_clean" LIKE '%ST. LOUIS%' OR "county_clean" LIKE '%SAINT LOUIS%';"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT "county_clean", COUNT(*) as polling_recordsFROM STG_POLLING_LOCATIONSGROUP BY "county_clean"ORDER BY "county_clean";"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT COUNTY FROM COUNTY_ANALYSIS WHERE COUNTY LIKE '%ST. LOUIS%';"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT COUNTY FROM COUNTY_POLLING_SUMMARY WHERE COUNTY LIKE '%ST. LOUIS%';"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT COUNTY, VOTES_2020 FROM COUNTY_ANALYSIS WHERE COUNTY LIKE '%ST. LOUIS%';"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT     a.COUNTY,    a.VOTES_2020,    p.UNIQUE_POLLING_PLACES,    ROUND(a.VOTES_2020 / NULLIF(p.UNIQUE_POLLING_PLACES, 0), 0) AS VOTERS_PER_POLLING_PLACEFROM COUNTY_ANALYSIS aLEFT JOIN COUNTY_POLLING_SUMMARY p     ON a.COUNTY = p.COUNTY     OR (a.COUNTY = 'ST. LOUIS COUNTY' AND p.COUNTY = 'ST. LOUIS')WHERE a.COUNTY LIKE '%ST. LOUIS%';"""df = pd.read_sql(sql, conn)df

In [ ]:
# Execute SQL Querysql = """SELECT * FROM STG_POLLING_LOCATIONS LIMIT 5;"""df = pd.read_sql(sql, conn)df